# Baseline Models and Standard Explanations — Four Core Datasets

This notebook trains the standard baseline models from the Chapter 6 *Model
Training Protocol* (`sec:model_training`) on the four core datasets used across
the contribution chapters — **Titanic, Pima Diabetes, COMPAS, and Adult
Census** — and produces, for each dataset, the conventional explanations those
models yield:

- **SHAP values** on the neural network (KernelSHAP, k-means-15 background,
  ~80 sampled test instances) — beeswarm plus an aggregated mean-|SHAP| bar;
- **Logistic-regression coefficients** — signed standardized $\hat\beta$;
- **Pruned decision tree** — the `max_depth=5`, `min_samples_leaf=20` tree;
- **Performance metrics** — train/test accuracy, precision, recall, F1, ROC-AUC.

Preprocessing (median-age imputation, the Adult 12-feature subset with grouped
categoricals, COMPAS ProPublica filtering, numeric typing) is reused *verbatim*
from the Chapter 7 SMD notebook (`Chapter 7 /potential_outcomes_SMD.ipynb`) so
the two chapters share one pipeline. The protocol follows Chapter 6: an 80/20
stratified split with `random_state=42`, continuous features standardized for
the LR and neural models, decision tree left on raw values.

> **Note (LR coverage).** Chapter 6 reports logistic regression only for
> Titanic and Pima Diabetes (Table `tab:model_performance`). Here LR is fitted
> on all four datasets for completeness; the COMPAS and Adult LR rows are
> therefore an extension beyond the reported table.
>
> **Note (Diabetes neural network).** On Pima Diabetes the small, imbalanced
> training set causes early stopping to halt after ~24 iterations, before the
> network learns the minority (diabetic) class, collapsing it to a near-constant
> negative predictor (recall $\approx 0.07$ despite AUC $\approx 0.79$). For this
> one dataset the network is therefore trained with `early_stopping=False`
> (`max_iter=1000`); the other three datasets keep the Chapter 6 early-stopping
> setting unchanged.

All figures are written to `./model_baseline_plots/<dataset>/` and the metrics
table to `./data/model_baseline_metrics.csv`, relative to this `Chapter 8 /`
directory.

In [1]:
# Imports and configuration
import io, zipfile, urllib.request
from pathlib import Path
import numpy as np, pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)
import shap

SEED = 42
DATA_DIR = Path('./data'); DATA_DIR.mkdir(exist_ok=True)
PLOT_DIR = Path('./model_baseline_plots'); PLOT_DIR.mkdir(exist_ok=True)

plt.rcParams.update({'figure.dpi': 110, 'savefig.bbox': 'tight',
                     'font.size': 10, 'axes.titlesize': 12})

In [2]:
# Dataset loaders (verbatim from the Chapter 7 SMD notebook).
# Each loader encodes the exact Chapter 6 preprocessing and caches the
# processed frame to ./data/<name>.csv for offline, reproducible re-runs.
def _cached(name, build):
    p = DATA_DIR / f'{name}.csv'
    if p.exists():
        return pd.read_csv(p)
    df = build(); df.to_csv(p, index=False)
    return df

def load_titanic():
    def _b():
        d = sns.load_dataset('titanic')
        d['sex']   = d['sex'].map({'male': 1, 'female': 0})
        d['class'] = d['class'].map({'Third': 3, 'Second': 2, 'First': 1})
        d['embarked'] = LabelEncoder().fit_transform(d['embarked'].astype(str))
        d['age'] = d['age'].fillna(d['age'].median())
        return d
    return _cached('titanic', _b)

def load_diabetes():
    def _b():  # Pima, via OpenML
        d = fetch_openml('diabetes', version=1, as_frame=True, parser='auto').frame.copy()
        d = d.rename(columns={'preg':'Pregnancies','plas':'Glucose','pres':'BloodPressure',
            'skin':'SkinThickness','insu':'Insulin','mass':'BMI',
            'pedi':'DiabetesPedigreeFunction','age':'Age'})
        d['Outcome'] = (d['class'] == 'tested_positive').astype(int)
        return d.drop(columns=['class'])
    return _cached('diabetes', _b)

def load_adult():
    def _b():
        r = fetch_openml(name='adult', version=2, as_frame=True, parser='auto').frame.copy()
        r = r.dropna().reset_index(drop=True)
        r['income'] = (r['class'].str.strip().str.replace('.', '', regex=False) == '>50K').astype(int)
        r = r.drop(columns=['class', 'fnlwgt'])
        r = r.rename(columns={'education-num':'education_num','capital-gain':'capital_gain',
            'capital-loss':'capital_loss','hours-per-week':'hours_per_week',
            'marital-status':'marital_status','native-country':'native_country'})
        r['married'] = (r['marital_status'] == 'Married-civ-spouse').astype(int)
        r = r.drop(columns=['marital_status'])
        r['relationship_grp'] = r['relationship'].map({'Husband':'spouse','Wife':'spouse',
            'Not-in-family':'independent','Unmarried':'independent',
            'Own-child':'dependent','Other-relative':'dependent'})
        r = r.drop(columns=['relationship'])
        high = {'Exec-managerial','Prof-specialty','Protective-serv','Tech-support'}
        low  = {'Other-service','Handlers-cleaners','Farming-fishing','Priv-house-serv',
                'Machine-op-inspct','Armed-Forces'}
        r['occupation_grp'] = r['occupation'].apply(
            lambda x: 'high' if x in high else ('low' if x in low else 'mid'))
        r = r.drop(columns=['occupation'])
        r['workclass_grp'] = r['workclass'].map({'Federal-gov':'government','Local-gov':'government',
            'State-gov':'government','Self-emp-inc':'self_employed','Self-emp-not-inc':'self_employed',
            'Private':'private','Without-pay':'private'})
        r = r.drop(columns=['workclass'])
        r['race_grp'] = r['race'].map({'White':'white','Black':'black',
            'Asian-Pac-Islander':'other_race','Amer-Indian-Eskimo':'other_race','Other':'other_race'})
        r = r.drop(columns=['race'])
        for c in ['age','education_num','capital_gain','capital_loss','hours_per_week']:
            r[c] = pd.to_numeric(r[c], errors='coerce')
        return r
    d = _cached('adult', _b)
    if 'native_us' not in d.columns and 'native_country' in d.columns:
        d['native_us'] = (d['native_country'] == 'United-States').astype(int)
    return d

def load_compas():
    def _b():
        url = ('https://raw.githubusercontent.com/propublica/compas-analysis/'
               'master/compas-scores-two-years.csv')
        r = pd.read_csv(url)
        r = r[(r['days_b_screening_arrest'] <= 30) & (r['days_b_screening_arrest'] >= -30) &
              (r['is_recid'] != -1) & (r['c_charge_degree'] != 'O') &
              (r['score_text'] != 'N/A')].copy()
        r['race_black']      = (r['race'] == 'African-American').astype(int)
        r['juv_any']         = ((r['juv_fel_count'] + r['juv_misd_count'] + r['juv_other_count']) > 0).astype(int)
        r['sex']             = (r['sex'] == 'Male').astype(int)
        r['c_charge_degree'] = (r['c_charge_degree'] == 'F').astype(int)
        cols = ['age','sex','race_black','priors_count','c_charge_degree','juv_any','two_year_recid']
        return r[cols].dropna().reset_index(drop=True)
    return _cached('compas', _b)

In [3]:
# Per-dataset configuration (the four core datasets).
# `raw`/`to_bin` are numeric source features (used as-is here); `cat` are
# categorical source features (one-hot encoded). Keys match the Chapter 7
# registry so the two notebooks describe the same feature sets.
DATASETS = {
 'titanic':  dict(loader=load_titanic, target='survived',
    raw=['sibsp', 'parch'], to_bin=['fare', 'age'],
    cat=['sex', 'pclass', 'embarked'], title='Titanic'),
 'diabetes': dict(loader=load_diabetes, target='Outcome',
    raw=[], to_bin=['Pregnancies','Glucose','BloodPressure','SkinThickness',
                    'Insulin','BMI','DiabetesPedigreeFunction','Age'],
    cat=[], title='Pima Diabetes', nn_early_stop=False),  # see title-cell note
 'compas':   dict(loader=load_compas, target='two_year_recid',
    raw=[], to_bin=['age', 'priors_count'],
    cat=['sex', 'race_black', 'c_charge_degree', 'juv_any'], title='COMPAS'),
 'adult':    dict(loader=load_adult, target='income',
    raw=[], to_bin=['age','education_num','capital_gain','capital_loss','hours_per_week'],
    cat=['sex','married','workclass_grp','occupation_grp','relationship_grp',
         'race_grp','native_us'], title='Adult Census'),
}

In [4]:
# Design matrix and split
def build_Xy(df, cfg):
    '''Design matrix over the source features: numeric as-is, categoricals
    one-hot (col=level). colmap maps each design column to its source.'''
    y = df[cfg['target']].astype(int).values
    num = list(cfg.get('raw', []) + cfg['to_bin']); cat = cfg['cat']
    Xnum = df[num].apply(pd.to_numeric, errors='coerce'); Xnum = Xnum.fillna(Xnum.median())
    blocks, colmap = [Xnum], {c: c for c in num}
    for c in cat:
        d = pd.get_dummies(df[c].astype('object'), prefix=c, prefix_sep='=').astype(float)
        blocks.append(d)
        for dc in d.columns:
            colmap[dc] = c
    return pd.concat(blocks, axis=1), y, num + cat, colmap

def split(X, y):
    return train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

In [5]:
# Fit the three baseline models and collect metrics
def _metrics(model, Xtr, ytr, Xte, yte):
    proba = model.predict_proba(Xte)[:, 1]
    pred  = model.predict(Xte)
    return dict(train_acc=accuracy_score(ytr, model.predict(Xtr)),
                test_acc =accuracy_score(yte, pred),
                precision=precision_score(yte, pred, zero_division=0),
                recall   =recall_score(yte, pred, zero_division=0),
                f1       =f1_score(yte, pred, zero_division=0),
                roc_auc  =roc_auc_score(yte, proba))

def fit_models(Xtr, ytr, Xte, yte, nn_early_stop=True):
    '''Decision tree on raw features; LR and MLP on standardized features.
    Returns fitted objects (+ the shared scaler) and a per-model metrics dict.
    nn_early_stop=False (diabetes only) trains the MLP to convergence instead
    of halting early; see the title-cell note.'''
    dt = DecisionTreeClassifier(max_depth=5, min_samples_leaf=20,
                                random_state=SEED).fit(Xtr, ytr)

    scaler = StandardScaler().fit(Xtr)
    Xtr_s, Xte_s = scaler.transform(Xtr), scaler.transform(Xte)

    lr = LogisticRegression(max_iter=1000, random_state=SEED).fit(Xtr_s, ytr)
    mlp = MLPClassifier(hidden_layer_sizes=(32, 16),
                        max_iter=(400 if nn_early_stop else 1000),
                        random_state=SEED, early_stopping=nn_early_stop).fit(Xtr_s, ytr)

    metrics = {'Decision Tree':       _metrics(dt, Xtr, ytr, Xte, yte),
               'Logistic Regression': _metrics(lr, Xtr_s, ytr, Xte_s, yte),
               'Neural Network':      _metrics(mlp, Xtr_s, ytr, Xte_s, yte)}
    return dt, lr, mlp, scaler, metrics

In [6]:
# Explanation plots (each saves a PNG to model_baseline_plots/<name>/)
def _outdir(name):
    d = PLOT_DIR / name; d.mkdir(parents=True, exist_ok=True); return d

def plot_shap(name, mlp, scaler, Xtr, Xte, colmap, source, rng, title):
    '''KernelSHAP on the MLP: beeswarm over design columns plus a
    mean-|SHAP| bar aggregated back to source features (Ch6 protocol).'''
    cols = list(Xtr.columns)
    Xtr_s, Xte_s = scaler.transform(Xtr), scaler.transform(Xte)
    bg  = shap.kmeans(Xtr_s, min(15, len(Xtr_s)))
    idx = rng.choice(len(Xte_s), min(80, len(Xte_s)), replace=False)
    expl = shap.KernelExplainer(lambda z: mlp.predict_proba(z)[:, 1], bg)
    sv = np.asarray(expl.shap_values(Xte_s[idx], nsamples='auto', silent=True))

    # Beeswarm: colour by the unscaled feature value for readability.
    disp = Xte.iloc[idx].reset_index(drop=True)
    shap.summary_plot(sv, features=disp, feature_names=cols, show=False,
                      max_display=min(20, len(cols)))
    fig = plt.gcf(); fig.suptitle(f'{title} — KernelSHAP (neural network)', y=1.02)
    fig.savefig(_outdir(name) / 'shap_beeswarm.png', bbox_inches='tight'); plt.close(fig)

    # Aggregated mean-|SHAP| per source feature.
    per_col = pd.Series(np.abs(sv).mean(axis=0), index=cols)
    agg = pd.Series(0.0, index=source)
    for c, v in per_col.items():
        agg[colmap[c]] += float(v)
    agg = agg.sort_values()
    fig, ax = plt.subplots(figsize=(6, max(3, 0.4 * len(agg))))
    ax.barh(agg.index, agg.values, color='#4C72B0')
    ax.set_xlabel('mean |SHAP value|'); ax.set_title(f'{title} — SHAP importance')
    fig.savefig(_outdir(name) / 'shap_bar.png', bbox_inches='tight'); plt.close(fig)

def plot_lr_coef(name, lr, cols, title):
    coef = pd.Series(lr.coef_[0], index=cols).sort_values()
    colors = ['#C44E52' if v < 0 else '#4C72B0' for v in coef.values]
    fig, ax = plt.subplots(figsize=(6, max(3, 0.4 * len(coef))))
    ax.barh(coef.index, coef.values, color=colors)
    ax.axvline(0, color='k', lw=0.8)
    ax.set_xlabel(r'standardized coefficient $\hat\beta_j$')
    ax.set_title(f'{title} — logistic-regression coefficients')
    fig.savefig(_outdir(name) / 'lr_coefficients.png', bbox_inches='tight'); plt.close(fig)

def plot_tree_fig(name, dt, cols, title):
    fig, ax = plt.subplots(figsize=(22, 12))
    plot_tree(dt, feature_names=cols, class_names=['neg', 'pos'], filled=True,
              rounded=True, impurity=False, proportion=True, fontsize=8, ax=ax)
    ax.set_title(f'{title} — decision tree (max_depth=5, min_samples_leaf=20)')
    fig.savefig(_outdir(name) / 'decision_tree.png', bbox_inches='tight'); plt.close(fig)

In [7]:
# Driver: run all four datasets
rng = np.random.default_rng(SEED)
rows = []

for name, cfg in DATASETS.items():
    print(f"\n{'#'*66}\n# {cfg['title']}\n{'#'*66}")
    df = cfg['loader']()
    X, y, source, colmap = build_Xy(df, cfg)
    print(f"n={len(df)}  design cols={X.shape[1]}  balance={dict(pd.Series(y).value_counts())}")

    Xtr, Xte, ytr, yte = split(X, y)
    dt, lr, mlp, scaler, metrics = fit_models(Xtr, ytr, Xte, yte,
                                              nn_early_stop=cfg.get('nn_early_stop', True))

    for model, m in metrics.items():
        print(f"  {model:20s}  train={m['train_acc']:.3f}  test={m['test_acc']:.3f}  "
              f"P={m['precision']:.3f}  R={m['recall']:.3f}  F1={m['f1']:.3f}  AUC={m['roc_auc']:.3f}")
        rows.append(dict(dataset=cfg['title'], model=model, **m))

    cols = list(X.columns)
    plot_tree_fig(name, dt, cols, cfg['title'])
    plot_lr_coef(name, lr, cols, cfg['title'])
    plot_shap(name, mlp, scaler, Xtr, Xte, colmap, source, rng, cfg['title'])
    print(f"  saved plots -> {PLOT_DIR / name}/")


##################################################################
# Titanic
##################################################################
n=891  design cols=13  balance={0: 549, 1: 342}
  Decision Tree         train=0.837  test=0.788  P=0.804  R=0.594  F1=0.683  AUC=0.816
  Logistic Regression   train=0.809  test=0.804  P=0.793  R=0.667  F1=0.724  AUC=0.844
  Neural Network        train=0.810  test=0.760  P=0.795  R=0.507  F1=0.619  AUC=0.822


  saved plots -> model_baseline_plots/titanic/

##################################################################
# Pima Diabetes
##################################################################
n=768  design cols=8  balance={0: 500, 1: 268}


  Decision Tree         train=0.801  test=0.766  P=0.725  R=0.537  F1=0.617  AUC=0.799
  Logistic Regression   train=0.792  test=0.714  P=0.609  R=0.519  F1=0.560  AUC=0.823
  Neural Network        train=0.915  test=0.708  P=0.588  R=0.556  F1=0.571  AUC=0.763


  saved plots -> model_baseline_plots/diabetes/

##################################################################
# COMPAS
##################################################################
n=6172  design cols=10  balance={0: 3363, 1: 2809}


  Decision Tree         train=0.688  test=0.672  P=0.659  R=0.580  F1=0.617  AUC=0.725
  Logistic Regression   train=0.676  test=0.682  P=0.696  R=0.534  F1=0.604  AUC=0.731
  Neural Network        train=0.679  test=0.685  P=0.707  R=0.525  F1=0.603  AUC=0.737


  saved plots -> model_baseline_plots/compas/

##################################################################
# Adult Census
##################################################################
n=45222  design cols=23  balance={0: 34014, 1: 11208}


  Decision Tree         train=0.845  test=0.839  P=0.811  R=0.459  F1=0.586  AUC=0.880
  Logistic Regression   train=0.846  test=0.841  P=0.722  R=0.583  F1=0.645  AUC=0.899
  Neural Network        train=0.855  test=0.846  P=0.732  R=0.596  F1=0.657  AUC=0.906


  saved plots -> model_baseline_plots/adult/


In [8]:
# Performance summary
metrics_df = pd.DataFrame(rows)[['dataset', 'model', 'train_acc', 'test_acc',
                                 'precision', 'recall', 'f1', 'roc_auc']]
out = DATA_DIR / 'model_baseline_metrics.csv'
metrics_df.to_csv(out, index=False)
print(f'saved {out}')
metrics_df.round(3)

saved data/model_baseline_metrics.csv


,dataset,model,train_acc,test_acc,precision,recall,f1,roc_auc
0,Titanic,Decision Tree,0.837,0.788,0.804,0.594,0.683,0.816
1,Titanic,Logistic Regression,0.809,0.804,0.793,0.667,0.724,0.844
2,Titanic,Neural Network,0.810,0.760,0.795,0.507,0.619,0.822
3,Pima Diabetes,Decision Tree,0.801,0.766,0.725,0.537,0.617,0.799
4,Pima Diabetes,Logistic Regression,0.792,0.714,0.609,0.519,0.560,0.823
5,Pima Diabetes,Neural Network,0.915,0.708,0.588,0.556,0.571,0.763
6,COMPAS,Decision Tree,0.688,0.672,0.659,0.580,0.617,0.725
7,COMPAS,Logistic Regression,0.676,0.682,0.696,0.534,0.604,0.731
8,COMPAS,Neural Network,0.679,0.685,0.707,0.525,0.603,0.737
9,Adult Census,Decision Tree,0.845,0.839,0.811,0.459,0.586,0.880
